# Recurrent Neural Networks
In this exercise, we will implement a simple one-layer recurrent neural network. We will use the formula for an [Elman RNN](https://en.wikipedia.org/wiki/Recurrent_neural_network#Elman_networks_and_Jordan_networks), one of the most basic and classical RNNs. The hidden state update and output at time $t$ are defined like this:

$$
\begin{align}
h_t &= \tanh(W_h x_t + U_h h_{t-1} + b_h) \\
y_t &= \tanh(W_y h_t + b_y)
\end{align}
$$

In [1]:
import torch
import torch.nn as nn

We start by defining the RNN as a subclass of `nn.Module`. The network's parameters are created in the `__init__` method. Use `input_dim`, `hidden_dim` and `output_dim` as arguments that define the dimensionality of the input/hidden/output vectors. Define your parameters as `nn.Parameter` with the appropriate dimensions. The documentation of `torch.nn` can be found [here](https://pytorch.org/docs/stable/nn.html).

In [2]:
class RNN(nn.Module):
    
    def __init__(self, input_dim, hidden_dim, output_dim):
        super().__init__()
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.output_dim = output_dim
        
        self.W_xh = nn.Parameter(torch.zeros(hidden_dim, input_dim))
        self.W_hh = nn.Parameter(torch.zeros(hidden_dim, hidden_dim))
        self.W_hy = nn.Parameter(torch.zeros(output_dim, hidden_dim))
        
        self.b_h = nn.Parameter(torch.zeros(hidden_dim))
        self.b_y = nn.Parameter(torch.zeros(output_dim))
        

Add a function `reset_parameters` that initializes your parameters. Pick a suitable distribution from [nn.init](https://pytorch.org/docs/stable/nn.init.html).

In [3]:
def reset_parameters(self):
    for weight in self.parameters():
        nn.init.normal_(weight, 0, 1)

RNN.reset_parameters = reset_parameters

Add a `forward` function that takes an input and a starting hidden state $h_{t-1}$ and returns the updated hidden state $h_t$ and output $y$ as outputs. The initial hidden state $h_0$ can be initialized randomly/to all zeros.

In [7]:
def forward(self, x, hidden_state):
    h_t = torch.tanh(self.W_xh @ x + self.W_hh @ hidden_state + self.b_h)
    y_t = torch.tanh(self.W_hy @ h_t + self.b_y)
    return y_t, h_t

RNN.forward = forward

Test your RNN with a single input.

In [8]:
input_dim = 5
hidden_dim = 20
output_dim = 10

rnn = RNN(input_dim, hidden_dim, output_dim)
rnn.reset_parameters()

x = torch.randn(input_dim)
h0 = torch.zeros(hidden_dim)

y, new_hidden_state = rnn(x, h0)

print(y, new_hidden_state)

tensor([-0.9533,  0.8893,  0.0693,  0.9191, -0.8608,  0.9783,  0.9952, -0.9988,
        -0.9919, -1.0000], grad_fn=<TanhBackward0>) tensor([-0.3241,  0.6253, -0.9503,  0.3531,  0.7698,  0.7991,  0.9793, -0.9705,
        -0.9994, -0.5560,  0.8702, -0.9995,  0.5229, -0.6359,  0.9884,  0.9970,
         0.8707, -0.9899, -0.9932,  0.9962], grad_fn=<TanhBackward0>)


Now create an input sequence and run it through your RNN.

In [9]:
seq_length = 4
inputs = [torch.randn(input_dim) for _ in range(seq_length)]
hidden_state = torch.zeros(hidden_dim)
outputs = []

for x in inputs:
    y, hidden_state = rnn(x, hidden_state)
    outputs.append(y)
    
print(outputs)

[tensor([ 0.8858, -0.9992, -0.3012,  0.9961, -0.9991, -0.9955,  1.0000,  0.7451,
        -1.0000,  1.0000], grad_fn=<TanhBackward0>), tensor([-1.0000,  0.9213, -0.3360, -0.9982, -0.9946,  0.9997,  0.8976, -0.9996,
        -0.7567, -0.9907], grad_fn=<TanhBackward0>), tensor([ 0.2065, -0.9999,  0.7712,  1.0000, -0.3945,  0.9297, -1.0000, -0.5450,
         0.8409,  0.9975], grad_fn=<TanhBackward0>), tensor([-0.8436, -1.0000,  0.9819,  0.8102, -0.9999, -1.0000,  0.5574,  0.9345,
        -1.0000,  1.0000], grad_fn=<TanhBackward0>)]


The final hidden state encodes all the information present in the input sequence. It can be used as a feature for classification, or to initialize a decoder RNN to do translation, for example.

Now look at PyTorch's documentation for the [`nn.RNN`](https://pytorch.org/docs/stable/generated/torch.nn.RNN.html) and the [`nn.RNNCell`](https://pytorch.org/docs/stable/generated/torch.nn.RNNCell.html) classes. What is the difference between the two? What is the difference to the definition from Wikipedia we used above? Run your input sequence through both the `nn.RNN` and the `nn.RNNCell`.